# LLM Inference Optimization: Quantization & Benchmarking

**Goal:** Take a pretrained language model (GPT-2), measure its baseline inference cost (latency, memory, quality), then apply quantization to reduce its footprint — and measure the trade-off precisely.

**Pipeline:**
1. Load baseline model
2. Benchmark: latency, GPU memory, perplexity
3. Apply 8-bit quantization (`bitsandbytes`)
4. Benchmark quantized model
5. Compare results with plots
6. Qualitative check: does generated text still make sense?

**Runtime:** Set Colab runtime to GPU (`Runtime > Change runtime type > T4 GPU`) before running.

## 1. Setup

In [ ]:
!pip install -q torch transformers datasets bitsandbytes accelerate matplotlib pandas

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

import time
import gc
import torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    BitsAndBytesConfig,
)
from datasets import load_dataset

assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > T4 GPU"
device = "cuda"
MODEL_NAME = "gpt2"  # 124M params. Try "gpt2-medium" (355M) if you want a bigger effect.
print("GPU:", torch.cuda.get_device_name(0))

## 2. Utility functions: memory + latency measurement

In [ ]:
def reset_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def get_gpu_memory_mb():
    return torch.cuda.max_memory_allocated() / (1024 ** 2)

def benchmark_latency(model, tokenizer, prompt="The future of artificial intelligence is",
                       max_new_tokens=50, n_runs=10, warmup=3):
    """Measures average wall-clock time to generate `max_new_tokens` tokens."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # warmup (first runs are slower due to CUDA kernel compilation/caching)
    for _ in range(warmup):
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)

    avg_time = sum(times) / len(times)
    ms_per_token = (avg_time / max_new_tokens) * 1000
    return {"avg_total_s": avg_time, "ms_per_token": ms_per_token, "raw_times": times}

def compute_perplexity(model, tokenizer, text, stride=512, max_length=1024):
    """Standard sliding-window perplexity computation on a block of text."""
    encodings = tokenizer(text, return_tensors="pt")
    seq_len = encodings.input_ids.size(1)

    nlls = []
    prev_end = 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = encodings.input_ids[:, begin:end].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss * trg_len

        nlls.append(neg_log_likelihood)
        prev_end = end
        if end == seq_len:
            break

    return torch.exp(torch.stack(nlls).sum() / end).item()

## 3. Load evaluation text (WikiText-2)

In [ ]:
wikitext = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
eval_text = "\n\n".join(wikitext["text"][:200])  # small slice, enough for a stable perplexity estimate
print(f"Eval text length: {len(eval_text)} characters")

## 4. Baseline model: load, benchmark

In [ ]:
reset_gpu()
tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
model_fp32 = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model_fp32.eval()

baseline_memory = None  # measured after a forward pass below

In [ ]:
reset_gpu()
baseline_latency = benchmark_latency(model_fp32, tokenizer)
baseline_memory = get_gpu_memory_mb()
baseline_ppl = compute_perplexity(model_fp32, tokenizer, eval_text)

print(f"Baseline (FP32):")
print(f"  Latency:    {baseline_latency['ms_per_token']:.2f} ms/token")
print(f"  Memory:     {baseline_memory:.1f} MB")
print(f"  Perplexity: {baseline_ppl:.2f}")

## 5. Free baseline model, load 8-bit quantized version
We must free the FP32 model from GPU memory before loading the quantized one, so the memory measurement is fair (isolated).

In [ ]:
del model_fp32
reset_gpu()

bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model_int8 = GPT2LMHeadModel.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map={"": 0})
model_int8.eval()

In [ ]:
reset_gpu()
int8_latency = benchmark_latency(model_int8, tokenizer)
int8_memory = get_gpu_memory_mb()
int8_ppl = compute_perplexity(model_int8, tokenizer, eval_text)

print(f"Quantized (INT8):")
print(f"  Latency:    {int8_latency['ms_per_token']:.2f} ms/token")
print(f"  Memory:     {int8_memory:.1f} MB")
print(f"  Perplexity: {int8_ppl:.2f}")

## 6. Results table

In [ ]:
results = pd.DataFrame([
    {"Model": "FP32 (baseline)", "Latency (ms/token)": baseline_latency["ms_per_token"],
     "Memory (MB)": baseline_memory, "Perplexity": baseline_ppl},
    {"Model": "INT8 (quantized)", "Latency (ms/token)": int8_latency["ms_per_token"],
     "Memory (MB)": int8_memory, "Perplexity": int8_ppl},
])
results["Memory reduction"] = f"{(1 - int8_memory / baseline_memory) * 100:.1f}%"
results["Latency change"] = f"{(int8_latency['ms_per_token'] / baseline_latency['ms_per_token'] - 1) * 100:+.1f}%"
results["PPL change"] = f"{(int8_ppl / baseline_ppl - 1) * 100:+.2f}%"

results.to_csv("results.csv", index=False)
results

## 7. Plots

In [ ]:
labels = ["FP32", "INT8"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(labels, [baseline_latency["ms_per_token"], int8_latency["ms_per_token"]],
            color=["#4C72B0", "#55A868"])
axes[0].set_title("Latency (ms/token)")
axes[0].set_ylabel("ms")

axes[1].bar(labels, [baseline_memory, int8_memory], color=["#4C72B0", "#55A868"])
axes[1].set_title("GPU Memory (MB)")
axes[1].set_ylabel("MB")

axes[2].bar(labels, [baseline_ppl, int8_ppl], color=["#4C72B0", "#55A868"])
axes[2].set_title("Perplexity (lower = better quality)")
axes[2].set_ylabel("PPL")

plt.tight_layout()
plt.savefig("comparison_plot.png", dpi=150)
plt.show()

## 8. Qualitative check: does generation still make sense?

In [ ]:
test_prompts = [
    "The future of artificial intelligence is",
    "In a small village, there lived a",
    "The capital of France is",
]

def generate(model, prompt, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True)

for p in test_prompts:
    print("PROMPT:", p)
    print("  INT8:", generate(model_int8, p))
    print()

## 9. Conclusions

Measured on GPT-2 (124M), T4 GPU (Colab):

| Model | Latency (ms/token) | Memory (MB) | Perplexity |
|---|---|---|---|
| FP32 (baseline) | 31.76 | 728.63 | 26.41 |
| INT8 (quantized) | 32.44 | 491.58 | 26.54 |

- **Memory reduction: -32.5%** — the main win of quantization here (728MB -> 492MB).
- **Latency change: +2.1%** — quantization did *not* speed up generation, it was marginally slower. On small models like GPT-2, the overhead of dequantizing INT8 weights back to float at inference time outweighs any compute savings. This is expected behavior, not a bug, and is worth reporting as-is rather than hidden.
- **Quality impact: +0.49% perplexity** — essentially unchanged. Generated completions (section 8) remained coherent and grammatical across all test prompts.
- **Takeaway:** for a small model like GPT-2, INT8 quantization is a memory optimization, not a speed optimization. It's worth applying when the deployment constraint is GPU memory (e.g. serving more model instances per GPU, or running on memory-limited hardware) — but it should not be adopted with the expectation of faster inference at this model scale. Quantization's latency benefits typically become more visible on larger models and with inference stacks that have INT8-optimized kernels (e.g. TensorRT).